# Day 7 · Exercise 5: Reduce Summaries to Final Output

**What you'll build:** `reduce_summaries(summaries: list[str], model: str) -> str` — a function that takes the list of per-chunk summaries produced by the map phase and sends them together in a single `ollama.chat()` call, returning one final coherent summary.

**Why it matters:** The reduce step is what turns a collection of disconnected chunk summaries into a real answer; without it your map-reduce pipeline delivers a list instead of a summary, and the whole pipeline is incomplete.

## Your Implementation

In [ ]:
import ollama

REDUCE_SYSTEM_PROMPT = (
    "You are a precise summarization assistant. "
    "You will receive several intermediate summaries, each covering a different "
    "section of a longer document. Synthesise them into one concise summary "
    "of 3-5 sentences that covers the document as a whole. "
    "Write plain prose only - no headings, no bullet points, no section labels. "
    "Do not introduce information that is not present in the provided summaries."
)


def reduce_summaries(summaries: list[str], model: str) -> str:
    """Combine a list of per-chunk summaries into one final synthesised summary.

    Joins all chunk summaries into a single block of text and sends them
    to the model in one ollama.chat() call using a synthesis-focused system
    prompt. This is the reduce phase of the map-reduce summarisation pipeline.

    Args:
        summaries: A list of short summary strings produced by the map phase,
                   one per document chunk.
        model: The Ollama model name to use (e.g. "llama3.2").

    Returns:
        A single string: the final synthesised summary of the whole document.

    Example:
        chunks = [
            "Solar capacity grew sharply in Europe during 2023.",
            "Offshore wind investment accelerated across the North Sea.",
        ]
        result = reduce_summaries(chunks, "llama3.2")
        # result is a single coherent paragraph covering both points
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

MODEL = "llama3.2"

# Six summaries totalling ~900 chars ensures synthesis is shorter than the combined input.
SAMPLE_SUMMARIES = [
    "Renewable energy capacity grew significantly in Europe during 2023, "
    "with solar installations reaching record levels in Germany and Spain.",
    "Investment in offshore wind projects accelerated across the North Sea, "
    "with the UK and Denmark committing to major new developments.",
    "Policy frameworks in the EU set binding targets for member states, "
    "requiring at least 42 percent of energy from renewables by 2030.",
    "Consumer adoption of electric vehicles surged across European markets, "
    "driven by expanded charging infrastructure and government purchase incentives.",
    "Energy storage deployments, particularly grid-scale batteries, more than doubled "
    "year-on-year as utilities sought to balance intermittent renewable supply.",
    "Carbon pricing schemes were extended and tightened in several countries, "
    "raising the cost of fossil fuel generation and accelerating coal plant retirements.",
]


def _run_checks():
    score, total = 0, 4

    # Check 1: function exists and is callable
    try:
        assert callable(reduce_summaries), 'reduce_summaries is not defined'
        print(f'{_PASS} Check 1/{total}: function exists and is callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: single-item list returns a non-empty string
    try:
        single = ["Solar power adoption surged worldwide in 2023."]
        result = reduce_summaries(single, MODEL)
        assert isinstance(result, str), f'expected str, got {type(result).__name__}'
        assert len(result) > 0, 'returned an empty string'
        print(f'{_PASS} Check 2/{total}: single-item list returns a non-empty string')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')

    # Check 3: multi-item list returns a non-empty string
    try:
        result = reduce_summaries(SAMPLE_SUMMARIES, MODEL)
        assert isinstance(result, str), f'expected str, got {type(result).__name__}'
        assert len(result) > 0, 'returned an empty string'
        print(f'{_PASS} Check 3/{total}: multi-item list returns a non-empty string')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')
        return

    # Check 4: output is shorter than the concatenated input (synthesis, not expansion)
    try:
        result = reduce_summaries(SAMPLE_SUMMARIES, MODEL)
        combined_len = sum(len(s) for s in SAMPLE_SUMMARIES)
        assert len(result) < combined_len, (
            f'output ({len(result)} chars) is not shorter than '
            f'the combined input ({combined_len} chars) — the model may be '
            f'elaborating rather than condensing'
        )
        print(f'{_PASS} Check 4/{total}: output is shorter than combined input '
              f'({len(result)} < {combined_len} chars)')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
        print(f'  {score}/{total} passed.')
    else:
        print(f'  {score}/{total} passed. Keep going!')


_run_checks()

## Bonus Challenge

On Day 8 you will learn about prompt templating — storing prompts as reusable
strings with placeholders. As a preview: try extracting the number of output
sentences (`3-5`) from `REDUCE_SYSTEM_PROMPT` into a parameter so callers can
control how long the final summary is:

```python
def reduce_summaries(
    summaries: list[str],
    model: str,
    max_sentences: int = 5,
) -> str:
    ...
```

Call `reduce_summaries(SAMPLE_SUMMARIES, "llama3.2", max_sentences=2)` and
compare the output length with the default. Does a tighter sentence budget
produce a more concise result?

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import ollama

REDUCE_SYSTEM_PROMPT = (
    "You are a precise summarization assistant. "
    "You will receive several intermediate summaries, each covering a different "
    "section of a longer document. Synthesise them into one concise summary "
    "of 3-5 sentences that covers the document as a whole. "
    "Write plain prose only - no headings, no bullet points, no section labels. "
    "Do not introduce information that is not present in the provided summaries."
)


def reduce_summaries(summaries: list[str], model: str) -> str:
    """Combine a list of per-chunk summaries into one final synthesised summary."""
    combined = "\n\n".join(summaries)
    messages = [
        {"role": "system", "content": REDUCE_SYSTEM_PROMPT},
        {"role": "user",   "content": combined},
    ]
    response = ollama.chat(model=model, messages=messages)
    return response["message"]["content"]
```

**Why this works:** Joining the summaries with `"\n\n"` gives the model a clear
visual boundary between each chunk summary, which helps it treat them as distinct
sections rather than one continuous passage. The reduce system prompt is
deliberately different from the map-phase prompt — it frames the input as
intermediate summaries from different sections and explicitly asks for synthesis,
which prevents the model from simply returning the first item verbatim. Because
chunk summaries are short (typically 100-300 characters each), the concatenated
input almost always fits in one context window, so reduce is always a single
model call regardless of how many chunks the map phase produced.
</details>